# Merged Cuba + Uganda — Train & Test (Colab + Kaggle)

Pool **Cuba** and **Uganda** into one dataset, split it into train/test, and train binary
sickle-cell classifiers (**Circular** vs **Elongated**) on the mix — across several ImageNet
backbones: **MobileNetV2, ResNet50, EfficientNetB0, VGG16, VGG19, ConvNeXt-Tiny** (+ **InceptionV3**).

- **Uganda Circular images are ignored on purpose** — so the *Circular* (healthy) class is Cuba-only,
  while *Elongated* (sickled) = Cuba Elongated + Uganda Elongated. The pool is class-imbalanced.
- **Equal training percentage from both datasets:** each country is split **80/20 separately**, then
  concatenated, so Cuba and Uganda each contribute 80% of their images to training.
- Augmentation is **geometric-only** (flip/rotate/zoom) — no colour/brightness jitter.
- Every backbone uses the **same data + split**: frozen ImageNet features + a small trainable head, 10 epochs.
  Each model applies its **own** `preprocess_input` internally, so images are decoded once as raw pixels.
- Headline metric = **overall test accuracy**, reported alongside **per-country test accuracy** (Cuba / Uganda).

> **Setup (one time):** dataset `abdulelahmandorah/srsi-scd` is pulled via `kagglehub`, which needs a token even though it's public.
> 1. kaggle.com -> **Settings -> API -> Create New Token** -> open `kaggle.json`.
> 2. Colab **Secrets** panel -> add `KAGGLE_USERNAME` and `KAGGLE_KEY`, toggle **Notebook access** on for both.
> 3. Runtime -> **GPU**, then **Run All**.

In [ ]:
import os, glob, numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

SEED = 42
tf.keras.utils.set_random_seed(SEED)

CFG = {
    'kaggle_dataset': 'abdulelahmandorah/srsi-scd',   # public Kaggle dataset (downloaded via kagglehub)
    'data_root':      '',                             # set automatically by the download cell below
    'results_dir':    '/content/results_merged_models',  # figures + weights (Colab-local; see note in cell 3)
    'img_size': 224,   # common input size for all backbones
    'batch':    32,
    'epochs':   10,
    'test_frac': 0.2,  # held out PER COUNTRY -> equal training % from both datasets
    'classes':  ['Circular', 'Elongated'],   # 0 = Circular (healthy), 1 = Elongated (sickled)
}

In [ ]:
# --- authenticate: read the Kaggle token from Colab Secrets (no typing in the notebook) ---
!pip -q install kagglehub
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    print('Kaggle credentials loaded from Colab Secrets.')
except Exception as e:
    print('Colab Secrets not found -- will show a login widget instead.', e)

# --- download the public dataset via kagglehub (cached after the first run) ---
import kagglehub
try:
    dl_path = kagglehub.dataset_download(CFG['kaggle_dataset'])
except Exception:
    kagglehub.login()   # fallback: type your Kaggle username + key in the widget
    dl_path = kagglehub.dataset_download(CFG['kaggle_dataset'])
print('Downloaded ->', dl_path)

# --- point data_root at the folder that actually holds the class dirs (handles any wrapper folder) ---
def find_data_root(base):
    for root, dirs, _ in os.walk(base):
        low = {d.lower() for d in dirs}
        if 'circular' in low and 'elongated' in low:
            return root
    return base
CFG['data_root'] = find_data_root(dl_path)

# --- output dirs (Colab-local; wiped when the runtime disconnects.
#     To keep them, mount Drive and set CFG['results_dir'] to a MyDrive path.) ---
os.makedirs(CFG['results_dir'], exist_ok=True)
FIG_DIR = os.path.join(CFG['results_dir'], 'figures'); os.makedirs(FIG_DIR, exist_ok=True)
WT_DIR  = os.path.join(CFG['results_dir'], 'weights');  os.makedirs(WT_DIR,  exist_ok=True)
print('Data    ->', CFG['data_root'])
print('Outputs ->', CFG['results_dir'])

In [ ]:
import re
# Find every cell image (.jpg only, so EDA/report .png figures in the folder are skipped).
paths = glob.glob(os.path.join(CFG['data_root'], '**', '*.jpg'), recursive=True)

def tag(p):
    """Infer (country, class) -- reads folder names first, then the filename as a fallback,
    so it works whether the layout is Class/Country/ folders or country-in-filename."""
    s = p.lower(); name = os.path.basename(s)
    # country
    if   'cuba' in s:               country = 'cuba'
    elif 'uganda' in s:             country = 'uganda'
    elif name.startswith('uganda'): country = 'uganda'
    elif re.match(r'^[ceo]\d', name): country = 'cuba'      # c####/e####/o#### = Cuba files
    else:                           country = None
    # class
    if   'elongated' in s:          cls = 'Elongated'
    elif 'circular' in s:           cls = 'Circular'
    elif country == 'uganda' and 'pos' in name: cls = 'Elongated'
    elif country == 'uganda' and 'neg' in name: cls = 'Circular'
    elif country == 'cuba' and name.startswith('e'): cls = 'Elongated'
    elif country == 'cuba' and name.startswith('c'): cls = 'Circular'
    else:                           cls = None              # 'o####' (Other) & unknowns -> excluded
    return country, cls

# Cuba: BOTH classes.  Uganda: Elongated ONLY (Circular ignored on purpose).
cuba_paths, cuba_labels, uganda_elong = [], [], []
for p in paths:
    c, k = tag(p)
    if c == 'cuba' and k is not None:
        cuba_paths.append(p); cuba_labels.append(CFG['classes'].index(k))
    elif c == 'uganda' and k == 'Elongated':      # ignore Uganda Circular on purpose
        uganda_elong.append(p)

cuba_labels = np.array(cuba_labels)
print(f"Found {len(paths)} jpgs under {CFG['data_root']}")
print(f"Cuba: {len(cuba_paths)} imgs  (Circular={int((cuba_labels==0).sum())}, Elongated={int((cuba_labels==1).sum())})")
print(f"Uganda Elongated: {len(uganda_elong)} imgs  (Uganda Circular ignored)")
assert len(cuba_paths) > 0 and len(uganda_elong) > 0, 'Paths empty -- check the dataset downloaded and the folder names.'

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
IMG = CFG['img_size']

# decode to RAW pixels [0,255] -- each model applies its own preprocess_input later, so one dataset fits all
def decode(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG, IMG))
    return tf.cast(img, tf.float32), label

# geometric-only augmentation (no colour/brightness -- that's the cross-country signal we care about)
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
], name='augment')

def make_ds(img_paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((list(img_paths), list(labels)))
    if training:
        ds = ds.shuffle(len(img_paths), seed=SEED)
    ds = ds.map(decode, num_parallel_calls=AUTOTUNE).batch(CFG['batch'])
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)   # test set is NOT shuffled -> predictions stay aligned with labels/country

# --- split EACH country 80/20 separately, then merge (equal training % from both) ---
c_tr_p, c_te_p, c_tr_y, c_te_y = train_test_split(
    cuba_paths, cuba_labels, test_size=CFG['test_frac'], stratify=cuba_labels, random_state=SEED)

uga_labels = np.ones(len(uganda_elong), dtype=int)   # all Uganda kept are Elongated (label 1)
u_tr_p, u_te_p, u_tr_y, u_te_y = train_test_split(
    uganda_elong, uga_labels, test_size=CFG['test_frac'], random_state=SEED)   # single class -> no stratify

# merge; keep a country tag for the test set so we can score Cuba and Uganda separately
train_paths  = list(c_tr_p) + list(u_tr_p)
train_labels = np.concatenate([c_tr_y, u_tr_y])
test_paths   = list(c_te_p) + list(u_te_p)
test_labels  = np.concatenate([c_te_y, u_te_y])
test_country = np.array(['Cuba'] * len(c_te_p) + ['Uganda'] * len(u_te_p))

train_ds = make_ds(train_paths, train_labels, training=True)
test_ds  = make_ds(test_paths,  test_labels)

print(f"TRAIN {len(train_paths)}  (Cuba {len(c_tr_p)} + Uganda {len(u_tr_p)}) | "
      f"Circular={int((train_labels==0).sum())}, Elongated={int((train_labels==1).sum())}")
print(f"TEST  {len(test_paths)}  (Cuba {len(c_te_p)} + Uganda {len(u_te_p)}) | "
      f"Circular={int((test_labels==0).sum())}, Elongated={int((test_labels==1).sum())}")

In [ ]:
from tensorflow.keras import applications as A

# name -> (constructor, its matching preprocess_input).  Delete any line you don't want.
MODELS = {
    'MobileNetV2':    (A.MobileNetV2,    A.mobilenet_v2.preprocess_input),
    'ResNet50':       (A.ResNet50,       A.resnet50.preprocess_input),
    'EfficientNetB0': (A.EfficientNetB0, A.efficientnet.preprocess_input),
    'VGG16':          (A.VGG16,          A.vgg16.preprocess_input),
    'VGG19':          (A.VGG19,          A.vgg19.preprocess_input),
    'ConvNeXtTiny':   (A.ConvNeXtTiny,   A.convnext.preprocess_input),
    'InceptionV3':    (A.InceptionV3,    A.inception_v3.preprocess_input),   # reference baseline
}

def build_model(name):
    ctor, prep = MODELS[name]
    base = ctor(include_top=False, weights='imagenet', input_shape=(IMG, IMG, 3), pooling='avg')
    base.trainable = False   # transfer learning: train the head only (set True to fine-tune)
    inp = tf.keras.Input((IMG, IMG, 3))
    x = prep(inp)                       # model-specific preprocessing, applied to raw [0,255] input
    x = base(x, training=False)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    m = tf.keras.Model(inp, out, name=name)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return m

is_cuba = test_country == 'Cuba'
is_uga  = test_country == 'Uganda'

results = {}
for name in MODELS:
    print(f"\n===== {name} =====")
    tf.keras.backend.clear_session()
    model = build_model(name)
    hist = model.fit(train_ds, validation_data=test_ds, epochs=CFG['epochs'], verbose=2)
    te_pred = (model.predict(test_ds, verbose=0).ravel() >= 0.5).astype(int)
    results[name] = {
        'history':   hist.history,
        'te_pred':   te_pred,
        'test_acc':  float((te_pred == test_labels).mean()),
        'cuba_acc':  float((te_pred[is_cuba] == test_labels[is_cuba]).mean()),
        'uga_acc':   float((te_pred[is_uga]  == test_labels[is_uga]).mean()),   # Uganda is all Elongated -> = recall
    }
    model.save_weights(os.path.join(WT_DIR, f'{name}.weights.h5'))
    print(f"{name}: test acc={results[name]['test_acc']:.3f}  "
          f"(Cuba={results[name]['cuba_acc']:.3f}, Uganda={results[name]['uga_acc']:.3f})")

In [ ]:
# --- test accuracy & loss curves across all models (validation_data = merged test set) ---
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for name, r in results.items():
    ax[0].plot(r['history']['val_accuracy'], label=name)
    ax[1].plot(r['history']['val_loss'],     label=name)
ax[0].set_title('Merged test accuracy'); ax[0].set_xlabel('epoch'); ax[0].legend(fontsize=8)
ax[1].set_title('Merged test loss');     ax[1].set_xlabel('epoch'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'curves_all.png'), dpi=150); plt.show()

In [ ]:
# --- summary table + bar chart: overall test accuracy, plus per-country (Cuba / Uganda) ---
import pandas as pd
summary = pd.DataFrame({
    'Test_acc':        {k: v['test_acc'] for k, v in results.items()},
    'Cuba_test_acc':   {k: v['cuba_acc'] for k, v in results.items()},
    'Uganda_test_acc': {k: v['uga_acc']  for k, v in results.items()},
}).sort_values('Test_acc', ascending=False)
print(summary.round(4))
summary.to_csv(os.path.join(CFG['results_dir'], 'model_comparison.csv'))

summary.plot(kind='bar', figsize=(10, 4.5))
plt.title('Merged Cuba+Uganda: overall vs per-country test accuracy')
plt.ylabel('accuracy'); plt.ylim(0, 1); plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'model_comparison.png'), dpi=150); plt.show()

In [ ]:
# --- merged test confusion matrices (full 2x2; reveals any collapse onto the majority Elongated class) ---
n = len(results); cols = 3; rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3.6*rows)); axes = axes.ravel()
for ax, (name, r) in zip(axes, results.items()):
    ConfusionMatrixDisplay(confusion_matrix(test_labels, r['te_pred'], labels=[0, 1]),
                           display_labels=CFG['classes']).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f"{name}  (acc={r['test_acc']:.3f})")
for ax in axes[n:]:
    ax.axis('off')
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'cm_merged_test_all.png'), dpi=150); plt.show()